In [1]:
#========================================================================
# Name: plot_goes_csapr_stats.ipynb
# Author: McKenna W. Stanford
# Author Contact: mckenna.stanford@pnnl.gov
# Date Created: 01/13/2025
#
# Utility: Plots GOES CTT/IR Tb and CSAPR statistics
#========================================================================

In [2]:
#===============================
# Imports
#===============================
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import glob
from matplotlib.collections import PatchCollection
from matplotlib.patches import Rectangle, Patch
from matplotlib.lines import Line2D
import matplotlib.gridspec as gridspec
import datetime
import matplotlib.colors as colors
import matplotlib as mpl
from pytz import utc
from matplotlib.lines import Line2D


import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import cartopy.io.img_tiles as cimgt
import shapely.geometry as sgeom
import matplotlib.patheffects as path_effects
import pandas as pd
import time
import warnings
import scipy
warnings.filterwarnings("ignore")
import os

/global/common/software/m1657/mckenna/flextrkr/lib/python3.11/site-packages/pyproj/__init__.py:89: UserWarning: pyproj unable to set database path.
  _pyproj_global_context_initialize()


## Functions

In [3]:
def find_nearest(array, value):
    array = np.asarray(array)
    idx = (np.abs(array - value)).argmin()
    return array[idx], idx

def to_datetime(date):
    """
    Converts a numpy datetime64 object to a python datetime object 
    Input:
      date - a np.datetime64 object
    Output:
      DATE - a python datetime object
    """
    timestamp = ((date - np.datetime64('1970-01-01T00:00:00'))
                 / np.timedelta64(1, 's'))
    
    return datetime.datetime.fromtimestamp(timestamp, datetime.UTC)

In [3]:
if False:
    csapr_path = '/global/cfs/projectdirs/m1657/avarble/cacti/3km/csapr/'
    csapr_files = sorted(glob.glob(csapr_path+'*.nc'))
    num_csapr_files = len(csapr_files)
    print('# of CSAPR files:',num_csapr_files)

In [4]:
csapr_path = '/pscratch/sd/m/mckenna/cacti/csapr_regridded/'
#csapr_path = '/global/cfs/projectdirs/m1657/avarble/cacti/Taranis/taranis_corcsapr2cfrppiqcM1_gridded.c1/'
csapr_files = sorted(glob.glob(csapr_path+'*.nc'))
num_csapr_files = len(csapr_files)
print('# of CSAPR files:',num_csapr_files)

# of CSAPR files: 399


In [5]:
goes_path = '/pscratch/sd/m/mckenna/cacti/satcorps_regridded/'
goes_files = sorted(glob.glob(goes_path+'*.nc'))
num_goes_files = len(goes_files)
print('# of GOES files:',num_goes_files)

# of GOES files: 11621


## Construct list of datetimes and timestamps that correspond to CSAPR files

In [6]:
if False:
    csapr_datetime = []
    csapr_timestamp = []
    csapr_day = []
    
    for ii in range(num_csapr_files):
        tmp_str = csapr_files[ii].split('/')[-1]
        tmp_str = tmp_str.split('_')[-1]
        tmp_str = tmp_str.split('.')
        tmp_str = tmp_str[0]
        tmp_date = tmp_str[0:8]
        tmp_time = tmp_str[8:]
        tmp_year = int(tmp_date[0:4])
        tmp_month = int(tmp_date[4:6])
        tmp_day = int(tmp_date[6:8])
        tmp_hour = int(tmp_time[0:2])
        tmp_min = int(tmp_time[2:4])
        tmp_sec = int(tmp_time[4:6])
        tmp_datetime = datetime.datetime(tmp_year,tmp_month,tmp_day,tmp_hour,tmp_min,tmp_sec,tzinfo=utc)
        tmp_day = datetime.datetime(tmp_year,tmp_month,tmp_day,tzinfo=utc)
        tmp_timestamp = int(tmp_datetime.timestamp())
        csapr_datetime.append(tmp_datetime)
        csapr_timestamp.append(tmp_timestamp)
        csapr_day.append(tmp_day)
        
    csapr_timestamp = np.array(csapr_timestamp)
    csapr_datetime = np.array(csapr_datetime)
    csapr_day = np.array(csapr_day)
    unique_csapr_day = np.unique(csapr_day)

In [11]:
csapr_datetime = []
csapr_timestamp = []
csapr_day = []

for ii in range(num_csapr_files):
    tmp_str = csapr_files[ii].split('/')[-1]
    tmp_str = tmp_str.split('_')[1].split('.')
    tmp_date = tmp_str[-2]
    tmp_time = tmp_str[-1]
    tmp_year = int(tmp_date[0:4])
    tmp_month = int(tmp_date[4:6])
    tmp_day = int(tmp_date[6:8])
    tmp_hour = int(tmp_time[0:2])
    tmp_min = int(tmp_time[2:4])
    tmp_sec = int(tmp_time[4:6])
    tmp_datetime = datetime.datetime(tmp_year,tmp_month,tmp_day,tmp_hour,tmp_min,tmp_sec,tzinfo=utc)
    tmp_day = datetime.datetime(tmp_year,tmp_month,tmp_day,tzinfo=utc)
    tmp_timestamp = int(tmp_datetime.timestamp())
    csapr_datetime.append(tmp_datetime)
    csapr_timestamp.append(tmp_timestamp)
    csapr_day.append(tmp_day)
    
csapr_timestamp = np.array(csapr_timestamp)
csapr_datetime = np.array(csapr_datetime)
csapr_day = np.array(csapr_day)
unique_csapr_day = np.unique(csapr_day)

In [12]:
unique_csapr_day

array([datetime.datetime(2018, 11, 29, 0, 0, tzinfo=<UTC>),
       datetime.datetime(2018, 12, 4, 0, 0, tzinfo=<UTC>),
       datetime.datetime(2018, 12, 5, 0, 0, tzinfo=<UTC>),
       datetime.datetime(2018, 12, 19, 0, 0, tzinfo=<UTC>),
       datetime.datetime(2019, 1, 22, 0, 0, tzinfo=<UTC>),
       datetime.datetime(2019, 1, 23, 0, 0, tzinfo=<UTC>),
       datetime.datetime(2019, 1, 25, 0, 0, tzinfo=<UTC>),
       datetime.datetime(2019, 1, 29, 0, 0, tzinfo=<UTC>),
       datetime.datetime(2019, 2, 8, 0, 0, tzinfo=<UTC>)], dtype=object)

## Construct list of datetimes and timestamps that correspond to GOES files

In [17]:
goes_datetime = []
goes_timestamp = []
goes_day = []
for ii in range(num_goes_files):
    tmp_str = goes_files[ii].split('/')[-1].split('.')
    tmp_date = tmp_str[-3]
    tmp_time = tmp_str[-2]
    tmp_year = int(tmp_date[0:4])
    tmp_month = int(tmp_date[4:6])
    tmp_day = int(tmp_date[6:8])
    tmp_hour = int(tmp_time[0:2])
    tmp_min = int(tmp_time[2:4])
    tmp_sec = int(tmp_time[4:6])
    tmp_datetime = datetime.datetime(tmp_year,tmp_month,tmp_day,tmp_hour,tmp_min,tmp_sec,tzinfo=utc)
    tmp_day = datetime.datetime(tmp_year,tmp_month,tmp_day,tzinfo=utc)
    tmp_timestamp = int(tmp_datetime.timestamp())
    goes_datetime.append(tmp_datetime)
    goes_timestamp.append(tmp_timestamp)
    goes_day.append(tmp_day)
    
goes_timestamp = np.array(goes_timestamp)
goes_datetime = np.array(goes_datetime)
goes_day = np.array(goes_day)
unique_goes_day = np.unique(goes_day)

In [20]:
#unique_goes_day

# Loop through CSAPR times and find corresponding GOES time, if it exists

In [21]:
goes_match_files = []
goes_match_timestamp = []
goes_match_datetime = []
goes_match_day = []

csapr_match_files = []
csapr_match_timestamp = []
csapr_match_datetime = []
csapr_match_day = []

for ii in range(len(csapr_timestamp)):
    time_diff = np.abs(csapr_timestamp[ii]-goes_timestamp)
    min_time = np.min(time_diff)
    if min_time < 60.:
        dumid = np.argmin(time_diff)
        #print(csapr_timestamp[ii],goes_timestamp[dumid])
        #print(csapr_datetime[ii],goes_datetime[dumid])
        goes_match_files.append(goes_files[dumid])
        goes_match_timestamp.append(goes_timestamp[dumid])
        goes_match_datetime.append(goes_datetime[dumid])
        
        csapr_match_files.append(csapr_files[ii])
        csapr_match_timestamp.append(csapr_timestamp[ii])
        csapr_match_datetime.append(csapr_datetime[ii])
        csapr_match_day.append(csapr_day[ii])
        
goes_match_files = np.asarray(goes_match_files)
goes_match_timestamp = np.asarray(goes_match_timestamp)
goes_match_datetime = np.asarray(goes_match_datetime)
goes_match_datetime64 = np.asarray([np.datetime64(goes_match_datetime[dd]) for dd in range(len(goes_match_datetime))])

csapr_match_files = np.asarray(csapr_match_files)
csapr_match_timestamp = np.asarray(csapr_match_timestamp)
csapr_match_datetime = np.asarray(csapr_match_datetime)
csapr_match_datetime64 = np.asarray([np.datetime64(csapr_match_datetime[dd]) for dd in range(len(csapr_match_datetime))])
csapr_match_day = np.asarray(csapr_match_day)

unique_csapr_match_day = np.unique(csapr_match_day)

In [26]:

print(len(csapr_match_datetime))

395


In [31]:
#for ii in range(91,len(unique_csapr_match_day)):
for ii in range(len(unique_csapr_match_day)):
    print('Date:',unique_csapr_match_day[ii])
    dumid = np.where(csapr_match_day == unique_csapr_match_day[ii])
    tmp_csapr_files = csapr_match_files[dumid]
    tmp_csapr_datetime = csapr_match_datetime[dumid]
    tmp_csapr_timestamp = csapr_match_timestamp[dumid]
    
    tmp_goes_files = goes_match_files[dumid]
    tmp_goes_datetime = goes_match_datetime[dumid]
    tmp_goes_timestamp = goes_match_timestamp[dumid]
    
    
    print('# of CSAPR files:',len(tmp_csapr_files))
    print('# of GOES files:',len(tmp_goes_files))
    
    goes_ir_tb = []
    goes_ctt = []
    goes_cth = []
    goes_opd = []
    goes_lwp_iwp = []
    goes_particle_size = []
    goes_phase = []
    csapr_col_max_ref = []
    #csapr_lwp = []
    #csapr_lwc_at_max_ref = []
    #csapr_lwc_lowest = []
    #csapr_ref_lowest = []
    csapr_time = []
    goes_time = []
    
    for jj in range(len(tmp_csapr_files)):
        ds = xr.open_dataset(tmp_csapr_files[jj])
        if jj == 0.:
            lon = ds['point_longitude'].values
            lon = lon[0,:,:]
            lat = ds['point_latitude'].values
            lat = lat[0,:,:]
        tmp_ref = ds['taranis_attenuation_corrected_reflectivity'].values.squeeze()
        #tmp_lwc = ds['lwc_combined'].values.squeeze()
        #tmp_rr = ds['taranis_rain_rate'].values.squeeze()
        tmp_z = ds['z'].values.squeeze()
        ny = len(ds['y'].values)
        nx = len(ds['x'].values)
        #tmp_lwc[tmp_lwc < 0.] = 0.0
        #masked_lwc = np.ma.masked_invalid(tmp_lwc)
        #tmp_lwp_masked = scipy.integrate.trapezoid(masked_lwc,tmp_z,axis=0)
        #tmp_lwp = tmp_lwp_masked.filled(fill_value=np.nan)


        #==============================================
        # Calculate the column-maximum reflectivity and
        # grab the index
        #==============================================
        tmp_csapr_col_max_ref = np.nanmax(tmp_ref, axis=0)  # Shape: (77, 77)
        
        # Replace NaNs in tmp_ref with -inf so that nanargmax works correctly
        ref_filled = np.where(np.isnan(tmp_ref), -np.inf, tmp_ref)
        
        # Get the altitude indices of the max reflectivity
        col_max_ref_arg = np.nanargmax(ref_filled, axis=0).astype(float)  # Convert to float
        
        # Mask where tmp_csapr_col_max_ref is NaN (i.e., no valid values in the column)
        invalid_mask = np.isnan(tmp_csapr_col_max_ref)
        col_max_ref_arg[invalid_mask] = np.nan  # Now this works correctly
        
        #==============================================
        # Get LWC at the column-maximum reflectivity
        #==============================================
        if False:
            # Initialize output array with NaNs
            tmp_lwc_at_max_ref = np.full((ny, nx), np.nan)
            
            # Mask for valid indices
            valid_mask = ~invalid_mask  # Ensures we only index valid columns
            
            # Convert valid indices to integer
            valid_indices = col_max_ref_arg[valid_mask].astype(int)
            
            # Generate y and x coordinate arrays
            y_indices, x_indices = np.nonzero(valid_mask)  # Get (y, x) positions where mask is True
            
            # Use advanced indexing correctly
            tmp_lwc_at_max_ref[y_indices, x_indices] = tmp_lwc[valid_indices, y_indices, x_indices]


        #================================================
        # Find the LWC and coincident reflectivity at
        # the lowest level within 1-km of the surface AGL
        #================================================
        if False:
            # Define height threshold (1 km AGL)
            height_threshold = 1000  # in meters
            
            # Get dimensions
            nz, ny, nx = tmp_lwc.shape  # nz = 45, ny = 77, nx = 77
            
            # Create output arrays filled with NaNs
            tmp_lwc_lowest = np.full((ny, nx), np.nan)
            tmp_ref_lowest = np.full((ny, nx), np.nan)
            
            # Identify valid altitude levels below 1 km (boolean mask, shape: [45])
            valid_below_1km = tmp_z <= height_threshold  # 1D mask
            
            # Iterate over each (y, x) point to find the lowest valid altitude
            for j in range(ny):
                for i in range(nx):
                    # Find indices where tmp_lwc is valid (not NaN and > 0) within 1 km
                    valid_indices = np.where(valid_below_1km & ~np.isnan(tmp_lwc[:, j, i]) & (tmp_lwc[:, j, i] > 0))[0]
            
                    if valid_indices.size > 0:  # If there's at least one valid point
                        lowest_idx = valid_indices[0]  # Get the lowest valid altitude
            
                        # Assign values at this lowest valid altitude
                        tmp_lwc_lowest[j, i] = tmp_lwc[lowest_idx, j, i]
                        tmp_ref_lowest[j, i] = tmp_ref[lowest_idx, j, i]



        
        #csapr_lwp.append(tmp_lwp)
        csapr_col_max_ref.append(tmp_csapr_col_max_ref)
        #csapr_lwc_at_max_ref.append(tmp_lwc_at_max_ref)
        #csapr_lwc_lowest.append(tmp_lwc_lowest)
        #csapr_ref_lowest.append(tmp_ref_lowest)
        csapr_time.append(ds['time'].values[0])
        ds.close()


        ds = xr.open_dataset(tmp_goes_files[jj])
        goes_ir_tb.append(ds['temperature_ir'].values.squeeze())
        goes_ctt.append(ds['cloud_top_temperature'].values.squeeze())
        goes_cth.append(ds['cloud_top_height'].values.squeeze())
        goes_opd.append(ds['cloud_visible_optical_depth'].values.squeeze())
        goes_lwp_iwp.append(ds['cloud_lwp_iwp'].values.squeeze())
        goes_particle_size.append(ds['cloud_particle_size'].values.squeeze())
        goes_phase.append(ds['cloud_phase'].values.squeeze())
        goes_time.append(ds['time'].values[0])
        ds.close()

        
    goes_ir_tb = np.asarray(goes_ir_tb)-273.15
    goes_ctt = np.asarray(goes_ctt)-273.15
    goes_cth = np.asarray(goes_cth)
    goes_opd = np.asarray(goes_opd)
    goes_lwp_iwp = np.asarray(goes_lwp_iwp)
    goes_phase = np.asarray(goes_phase)
    goes_particle_size = np.asarray(goes_particle_size)
    goes_time = np.asarray(goes_time)
    #csapr_lwp = np.asarray(csapr_lwp)
    csapr_col_max_ref = np.asarray(csapr_col_max_ref)
    #csapr_lwc_at_max_ref = np.asarray(csapr_lwc_at_max_ref)
    #csapr_lwc_lowest = np.asarray(csapr_lwc_lowest)
    #csapr_ref_lowest = np.asarray(csapr_ref_lowest)
    csapr_time = np.asarray(csapr_time)
    csapr_epoch = np.asarray([int(to_datetime(csapr_time[dd]).timestamp()) for dd in range(len(csapr_time))])
    goes_epoch = np.asarray([int(to_datetime(goes_time[dd]).timestamp()) for dd in range(len(goes_time))])

    #==================================================
    # Create Xarray dataset and write to NetCDF file
    #==================================================
    

    # Make dictionary with output variables
    out_dict = {'col_max_ref':csapr_col_max_ref,\
                #'lwc_at_max_ref':csapr_lwc_at_max_ref,
                #'lwc_lowest':csapr_lwc_lowest,
                #'ref_lowest':csapr_ref_lowest,
                #'csapr_lwp':csapr_lwp,
                'ir_tb':goes_ir_tb,\
                'ctt':goes_ctt,\
                'cth':goes_cth,\
                'opd':goes_opd,\
                'lwp_iwp':goes_lwp_iwp,\
                'particle_size':goes_particle_size,\
                'phase':goes_phase,\
                'goes_time':goes_epoch}

    # Define variable attributes
    out_dict_attrs = {
                'col_max_ref': {
                    'long_name': 'Column-Maximum Radar Reflectivity',
                    'units': 'dBZ',
                },
                #'csapr_lwp': {
                #    'long_name': 'Liquid Water Path (essentially RWP)',
                #    'units': 'g/m^2',
                #},
                #'lwc_at_max_ref': {
                #    'long_name': 'LWC @ Column-Maximum Radar Reflectivity',
                #    'units': 'g/m^3',
                #},
                #'lwc_lowest': {
                #    'long_name': 'Lowest-level LWC within 1-km AGL',
                #    'units': 'g/m^3',
                #},
                #'ref_lowest': {
                #    'long_name': 'Reflectivity at lowest-level LWC within 1-km AGL',
                #    'units': 'dBZ',
                #},
                'ir_tb':{
                    'long_name': 'Infrared Brightness Temperature',
                    'units':'deg C'
                },
                'ctt':{
                    'long_name': 'Cloud-top Temperature',
                    'units':'deg C'
                },
                'cth':{
                    'long_name': 'Cloud-top Height',
                    'units':'km'
                },
                'opd':{
                    'long_name': 'Visible Optical Depth',
                    'units':'unitless',
                },
                'particle_size':{
                    'long_name': 'Effective Particle Radius or Diameter',
                    'units':'um',
                    'description_1':'If cloud_phase = water, this parameter is radius',
                    'description_2':'If cloud_phase = ice, this parameter is diameter',
                },
                'lwp_iwp':{
                    'long_name': 'Liquid or ice water path',
                    'units':'g/m^2',
                    'description_1':'If cloud_phase = water, this parameter is radius',
                    'description_2':'If cloud_phase = ice, this parameter is diameter',
                },
                'phase':{
                    'long_name': 'Cloud phase',
                    'units':'0=clear with snow/ice, 1=water, 2=ice, 3=no retrieval, 4=clear, 5=bad retrieval, 6=weak water, 7=weak ice',
                },
                'goes_time':{
                    'long_name': 'Time of GOES retrieval',
                    'units':'Epoch',
                    'description':'Since time grid corresponds to CSAPR times, this holds the GOES retrieval time that is require to be within 1 minutes of a given CSAPR time'
                }
    }   

    # Output parameters
    output_date = str(unique_csapr_match_day[ii].year)+str(unique_csapr_match_day[ii].month).zfill(2)+str(unique_csapr_match_day[ii].day).zfill(2)
    output_path = '/pscratch/sd/m/mckenna/cacti/matched_csapr_goes/2.5km/'
    output_filename = f'{output_path}CSAPR_GOES_matched_times_2.5km_{output_date}.nc'


    # Define dimensions
    time_dimname = 'time'
    lon_dimname = 'west_east'
    lat_dimname = 'south_north'

    var_dict = {}
    # Define output variable dictionary
    for key, value in out_dict.items():
        if np.ndim(value) == 1:
            var_dict[key] = ([time_dimname],value,out_dict_attrs[key])
        if np.ndim(value) == 3.:
            var_dict[key] = ([time_dimname,lat_dimname,lon_dimname],value,out_dict_attrs[key])

    # Define coordinate attributes
    coord_attr_dict = {'time':{'long_name':'CSAPR Epoch time','units':'Epoch'},
                       'lon':{'long_name':'Longitude','units':'degrees'},
                       'lat':{'long_name':'Latitude','units':'degrees'},
                      }

    # Define coordinates
    coord_dict = {
            'time': ([time_dimname],csapr_epoch,coord_attr_dict['time']),
            'lon': ([lat_dimname,lon_dimname],lon,coord_attr_dict['lon']),
            'lat': ([lat_dimname,lon_dimname],lat,coord_attr_dict['lat']),
            }

    # Define global attributes
    gattr_dict = {
        'title':  'Matched-time GOES & CSAPR datasets coarse-grained to 2.5 km for CACTI campaign (10-15-2018 - 03/03/2019)', \
        'Institution': 'Pacific Northwest National Laboratoy', \
        'Contact': 'McKenna Stanford, mckenna.stnaford@pnnl.gov', \
        'Created_on':  time.ctime(time.time()), \
        'source_csapr_path': csapr_path, \
        'source_goes_path': goes_path, \
        'Date': unique_csapr_match_day[ii].strftime('%Y%m%d'), \
    }

    # Define xarray dataset
    dsout = xr.Dataset(var_dict, coords=coord_dict, attrs=gattr_dict)

    # Delete file if it already exists
    if os.path.isfile(output_filename):
        os.remove(output_filename)

    # Set encoding/compression for all variables
    comp = dict(zlib=True)
    encoding = {var: comp for var in dsout.data_vars}

    # Write to netcdf file
    dsout.to_netcdf(path=output_filename, mode="w",
                    format="NETCDF4", unlimited_dims=time_dimname, encoding=encoding)
    
    print('')
    #print(aaaaa)

Date: 2018-11-29 00:00:00+00:00
# of CSAPR files: 48
# of GOES files: 48

Date: 2018-12-04 00:00:00+00:00
# of CSAPR files: 48
# of GOES files: 48

Date: 2018-12-05 00:00:00+00:00
# of CSAPR files: 48
# of GOES files: 48

Date: 2018-12-19 00:00:00+00:00
# of CSAPR files: 48
# of GOES files: 48

Date: 2019-01-22 00:00:00+00:00
# of CSAPR files: 48
# of GOES files: 48

Date: 2019-01-23 00:00:00+00:00
# of CSAPR files: 48
# of GOES files: 48

Date: 2019-01-25 00:00:00+00:00
# of CSAPR files: 29
# of GOES files: 29

Date: 2019-01-29 00:00:00+00:00
# of CSAPR files: 48
# of GOES files: 48

Date: 2019-02-08 00:00:00+00:00
# of CSAPR files: 30
# of GOES files: 30



In [32]:
#ds